# Notebook 01 — Stage M: SARIMA Forecasting

This notebook implements Stage M of the M-S-O pipeline:
1. Generate synthetic blood request demand for 12 hospitals (6 training days + 1 test day)
2. Fit independent SARIMA models per hospital via AIC grid search
3. Forecast 24-hour arrival rates (lambda_hat)
4. Evaluate forecast accuracy against held-out test day
5. Save outputs for Stages O and S

**All data is synthetic. No real hospital data is used.**

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on path
repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.utils.config_loader import load_config
from src.data_gen.generator import SyntheticDemandGenerator
from src.forecasting.sarima_model import SARIMAForecaster

print('Imports successful')

## 1. Load Configuration

In [ ]:
cfg = load_config(repo_root / 'config' / 'sarima.yaml')
gen_cfg = cfg['data_gen']
sarima_cfg = {
    'p_range': cfg['model']['p_range'],
    'd_range': cfg['model']['d_range'],
    'q_range': cfg['model']['q_range'],
    'P_range': cfg['model']['P_range'],
    'D_range': cfg['model']['D_range'],
    'Q_range': cfg['model']['Q_range'],
    's': cfg['model']['s'],
    'forecast': cfg['forecast'],
    'stationarity': cfg['stationarity'],
}
print('Config loaded')
print(f"H={gen_cfg['H']}, T={gen_cfg['T']}, D_train={gen_cfg['D_train']}, D_test={gen_cfg['D_test']}")

## 2. Generate Synthetic Demand

In [ ]:
generator = SyntheticDemandGenerator(gen_cfg)
data = generator.generate()

print(f"counts shape:      {data['counts'].shape}")
print(f"lambda_true shape: {data['lambda_true'].shape}")
print(f"train_counts shape:{data['train_counts'].shape}")
print(f"test_counts shape: {data['test_counts'].shape}")
print(f"Min lambda_true:   {data['lambda_true'].min():.4f} (floor=0.10)")

In [ ]:
# Save synthetic data
out_synthetic = repo_root / 'data' / 'synthetic'
generator.save(data, out_synthetic)
print(f'Synthetic data saved to {out_synthetic}')

### Visualize base demand pattern

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharey=True)
axes = axes.flatten()
hours = np.arange(24)
for h in range(12):
    ax = axes[h]
    # Plot all training days
    for d in range(6):
        ax.plot(hours, data['train_counts'][d, h, :], alpha=0.4, color='steelblue', lw=0.8)
    ax.plot(hours, data['lambda_true'][:6, h, :].mean(axis=0), 'r-', lw=2, label='Mean λ_true')
    ax.set_title(f'Hospital {h+1:02d}')
    ax.set_xlabel('Hour')
    if h % 4 == 0:
        ax.set_ylabel('Requests')
    if h == 0:
        ax.legend(fontsize=8)
fig.suptitle('Synthetic Training Demand (6 days, 12 hospitals)', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Fit SARIMA Models

In [ ]:
print('Fitting SARIMA models (one per hospital via AIC grid search)...')
print('This may take 3-8 minutes.')
print()

forecaster = SARIMAForecaster(sarima_cfg)
forecaster.fit(data['train_counts'])

n_converged = sum(m is not None for m in forecaster._fitted_models)
print(f'\nConvergence: {n_converged}/12 hospitals')

## 4. Generate Forecasts (lambda_hat)

In [ ]:
lambda_hat = forecaster.predict(steps=24)
print(f'lambda_hat shape: {lambda_hat.shape}')
print(f'Min: {lambda_hat.min():.4f}, Max: {lambda_hat.max():.4f}')
print(f'All >= 0.10: {(lambda_hat >= 0.10).all()}')

In [ ]:
# Save lambda_hat
out_processed = repo_root / 'data' / 'processed'
out_processed.mkdir(parents=True, exist_ok=True)
forecaster.save(out_processed / 'lambda_hat.csv')
print(f'lambda_hat saved to {out_processed / "lambda_hat.csv"}')

## 5. Forecast Evaluation

In [ ]:
# Ground truth for test day
lambda_true_test = data['lambda_true'][gen_cfg['D_train']]  # shape (12, 24)

metrics_df = forecaster.evaluate(lambda_true_test)
print('Forecast metrics per hospital:')
print(metrics_df.to_string(index=False))
print(f"\nMean MAE:  {metrics_df['MAE'].mean():.4f}")
print(f"Mean RMSE: {metrics_df['RMSE'].mean():.4f}")
print(f"Mean MAPE: {metrics_df['MAPE'].mean():.2f}%")
print(f"Mean Bias: {metrics_df['Bias'].mean():.4f}")

In [ ]:
# Save forecast metrics
metrics_df.to_csv(out_processed / 'forecast_metrics.csv', index=False)
print(f'Forecast metrics saved to {out_processed / "forecast_metrics.csv"}')

### Forecast vs Ground Truth (Fig 1 preview)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
axes = axes.flatten()
hours = np.arange(24)
for h in range(12):
    ax = axes[h]
    ax.plot(hours, lambda_true_test[h], 'k--', lw=1.5, label='True λ')
    ax.plot(hours, lambda_hat[h], 'r-', lw=1.5, label='λ̂ SARIMA')
    ax.set_title(f'Hospital {h+1:02d}', fontsize=9)
    ax.set_xlabel('Hour', fontsize=8)
    if h % 4 == 0:
        ax.set_ylabel('Requests/hr', fontsize=8)
    if h == 0:
        ax.legend(fontsize=7)
fig.suptitle('SARIMA Forecast vs Ground Truth — Test Day', fontsize=13)
plt.tight_layout()
plt.show()

## Stage M Complete

**Outputs saved:**
- `data/synthetic/demand_train.csv`
- `data/synthetic/demand_test.csv`
- `data/synthetic/lambda_true.csv`
- `data/processed/lambda_hat.csv` ← input to Stage O and Stage S
- `data/processed/forecast_metrics.csv`

**Next:** Run `notebook_02_optimization.ipynb`